### POS Tag

In [ ]:
from nltk import pos_tag
from nltk.tokenize import word_tokenize

In [ ]:
sentence = "Apple Inc. was founded by Steve Jobs in Cupertino, California."
tokens = word_tokenize(sentence)

tagged = pos_tag(tokens)
for word, tag in tagged:
    print(f"{word} : {tag}")

Apple : NNP
Inc. : NNP
was : VBD
founded : VBN
by : IN
Steve : NNP
Jobs : NNP
in : IN
Cupertino : NNP
, : ,
California : NNP
. : .


In [ ]:
from nltk import ne_chunk

sentence = "Apple Inc. was founded by Steve Jobs in Cupertino, California."
tokens = word_tokenize(sentence)
tagged = pos_tag(tokens)
entity_tree = ne_chunk(tagged) # ne_chunk takes POS-tagged tokens
print(entity_tree)
print(type(entity_tree))

(S
  (PERSON Apple/NNP)
  (ORGANIZATION Inc./NNP)
  was/VBD
  founded/VBN
  by/IN
  (PERSON Steve/NNP Jobs/NNP)
  in/IN
  (GPE Cupertino/NNP)
  ,/,
  (GPE California/NNP)
  ./.)
<class 'nltk.tree.tree.Tree'>


In [ ]:
def extract_enities(sentence):
    tokens = word_tokenize(sentence)
    tagged = pos_tag(tokens)
    entity_tree = ne_chunk(tagged)

    entities = []
    for subtree in entity_tree:
        if hasattr(subtree , "label"):
            entity_text = " ".join(word for word,tag in subtree.leaves())
            entities.append((entity_text, subtree.label()))
    return entities

sentence = "Apple Inc. was founded by Steve Jobs in Cupertino, California."
extract_enities(sentence)

[('Apple', 'PERSON'),
 ('Inc.', 'ORGANIZATION'),
 ('Steve Jobs', 'PERSON'),
 ('Cupertino', 'GPE'),
 ('California', 'GPE')]

In [ ]:
data = { 1 : "I am working at GOOGLE",
         2 : "My workspace Mcdonalds has few opeaning"
}

def extract_enities(sentence):
    tokens = word_tokenize(sentence)
    tagged = pos_tag(tokens)
    entity_tree = ne_chunk(tagged)

    entities = []
    for subtree in entity_tree:
        if hasattr(subtree , "label"):
            entity_text = " ".join(word for word,tag in subtree.leaves())
            # if subtree.label() == 'ORGANIZATION':
            entities.append((entity_text, subtree.label()))
    return entities

for k, v in data.items():
    e = extract_enities(v)
    print(e)

[('GOOGLE', 'ORGANIZATION')]
[('Mcdonalds', 'PERSON')]


# With Spacy

In [ ]:
import spacy


In [ ]:
nlp = spacy.load("en_core_web_sm")

data = {
    1: "I am working at GOOGLE", 
    2: "My workspace Mcdonalds has few opeaning"
}

def extract_entities(sentence):
    doc = nlp(sentence)
    # Extract the text and the label for each entity found
    return [(ent.text, ent.label_) for ent in doc.ents]

for k, v in data.items():
    e = extract_entities(v)
    print(f"Key {k}: {e}")

Key 1: [('GOOGLE', 'ORG')]
Key 2: [('Mcdonalds', 'FAC')]


# With Transformers

In [ ]:
# !pip install transformers 
# !pip install torch
# %pip install torch

Note: you may need to restart the kernel to use updated packages.


In [1]:
from transformers import pipeline

d:\Padai\Python\Krish\workshop\level\level-2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
print(torch.__version__)

from transformers import pipeline
ner_pipeline = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple",
)

2.14.1+cpu


d:\Padai\Python\Krish\workshop\level\level-2\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\Padai\Python\Krish\workshop\level\level-2\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `hugging

In [3]:
data = {
    1: "I am working at GOOGLE", 
    2: "My workspace Mcdonalds has few opeaning"
}

def extract_entities_transformer(sentence):
    results = ner_pipeline(sentence)
    # Extract entity text and their grouped labels (e.g., ORG)
    return [(res['word'], res['entity_group']) for res in results]

for k, v in data.items():
    print(f"Key {k}: {extract_entities_transformer(v)}")

Key 1: [('GOOGLE', 'ORG')]
Key 2: [('Mcdonalds', 'ORG')]


# ONE-HOT ENCODING

In [4]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are great pets",
]

# cat, sat, mat, dog, pets, great
# and, are, the, the --> too common

In [5]:
import pandas as pd

In [6]:
words = ["cat", "dog", "cat", "bird"]
df_words = pd.DataFrame({"word": words})

one_hot = pd.get_dummies(df_words["word"]).astype(int)
print(one_hot)

   bird  cat  dog
0     0    1    0
1     0    0    1
2     0    1    0
3     1    0    0


In [ ]:
# # Problem with OHE
# - 1. One Vector Per Word, not per document
# - 2. No notion of frequency
# - 3. Massive, sparse vectors at real world vocabulary sizes

# BAG OF WORDS

In [7]:
# Bag of words directly addresses the problem : No notion of frequency that OHE has

corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are great pets",
]

# corpus[0] --> []

# "dog bites man"
# "man bites dog"
# -> above will generate same BOW(BAG of WORDS) vector
# mat, log, pets, great

# CORPUS LEVEL

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(corpus)

vocab = vectorizer.get_feature_names_out()
print("Vocabulary:", vocab)

bow_df = pd.DataFrame(bow_matrix.toarray(), columns=vocab)
bow_df

Vocabulary: ['and' 'are' 'cat' 'cats' 'dog' 'dogs' 'great' 'log' 'mat' 'on' 'pets'
 'sat' 'the']


,and,are,cat,cats,dog,dogs,great,log,mat,on,pets,sat,the
0,0,0,1,0,0,0,0,0,1,1,0,1,2
1,0,0,0,0,1,0,0,1,0,1,0,1,2
2,1,1,0,1,0,1,1,0,0,0,1,0,0


In [9]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are great pets",
    "dog bites man",
    "man bites dog",
    "man can be a dog"
]


problems :
1. cat / cats have two seperate columns --> solution : lemma
2. BOW throws away word order -->
Solution for 2.

We cannot sort the columns alphabetically, but what we can do is, keep the words together and count them

"The", "cat" --> "The cat"

"cat", "sat" --> "cat sat"

In [10]:
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
bigram_matrix = bigram_vectorizer.fit_transform(corpus)

bigram_vocab = bigram_vectorizer.get_feature_names_out()
print("Bigram vocabulary:", bigram_vocab)

pd.DataFrame(bigram_matrix.toarray(), columns=bigram_vocab)

Bigram vocabulary: ['and dogs' 'are great' 'be dog' 'bites dog' 'bites man' 'can be'
 'cat sat' 'cats and' 'dog bites' 'dog sat' 'dogs are' 'great pets'
 'man bites' 'man can' 'on the' 'sat on' 'the cat' 'the dog' 'the log'
 'the mat']


,and dogs,are great,be dog,bites dog,bites man,can be,cat sat,cats and,dog bites,dog sat,dogs are,great pets,man bites,man can,on the,sat on,the cat,the dog,the log,the mat
0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,0,1
1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,1,1,0
2,1,1,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
5,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0


In [11]:
# include both unigram and bigram
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
bigram_matrix = bigram_vectorizer.fit_transform(corpus)

bigram_vocab = bigram_vectorizer.get_feature_names_out()
print("Bigram vocabulary:", bigram_vocab)

pd.DataFrame(bigram_matrix.toarray(), columns=bigram_vocab)

Bigram vocabulary: ['and' 'and dogs' 'are' 'are great' 'be' 'be dog' 'bites' 'bites dog'
 'bites man' 'can' 'can be' 'cat' 'cat sat' 'cats' 'cats and' 'dog'
 'dog bites' 'dog sat' 'dogs' 'dogs are' 'great' 'great pets' 'log' 'man'
 'man bites' 'man can' 'mat' 'on' 'on the' 'pets' 'sat' 'sat on' 'the'
 'the cat' 'the dog' 'the log' 'the mat']


,and,and dogs,are,are great,be,be dog,bites,bites dog,bites man,can,...,on,on the,pets,sat,sat on,the,the cat,the dog,the log,the mat
0,0,0,0,0,0,0,0,0,0,0,...,1,1,0,1,1,2,1,0,0,1
1,0,0,0,0,0,0,0,0,0,0,...,1,1,0,1,1,2,0,1,1,0
2,1,1,1,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
3,0,0,0,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,1,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [12]:
# include both unigram and bigram
bigram_vectorizer = CountVectorizer(ngram_range=(3, 3))
bigram_matrix = bigram_vectorizer.fit_transform(corpus)

bigram_vocab = bigram_vectorizer.get_feature_names_out()
print("Bigram vocabulary:", bigram_vocab)

pd.DataFrame(bigram_matrix.toarray(), columns=bigram_vocab)

Bigram vocabulary: ['and dogs are' 'are great pets' 'can be dog' 'cat sat on' 'cats and dogs'
 'dog bites man' 'dog sat on' 'dogs are great' 'man bites dog'
 'man can be' 'on the log' 'on the mat' 'sat on the' 'the cat sat'
 'the dog sat']


,and dogs are,are great pets,can be dog,cat sat on,cats and dogs,dog bites man,dog sat on,dogs are great,man bites dog,man can be,on the log,on the mat,sat on the,the cat sat,the dog sat
0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0
1,0,0,0,0,0,0,1,0,0,0,1,0,1,0,1
2,1,1,0,0,1,0,0,1,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
5,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0


problem with BOW/ngrams :
1. every word counts equally

ngrams(1,2) --> the --> 2 --> remove this with stop word --> fix the issue

corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are great pets",
]

cat, sat, mat, dog, pets, great
and, are, the, the --> too common

# TF-IDF

- TF - how often a word appreas in this document  (Term frequency)
- IDF - how rare that word is across the whole corpus (Inverse document frequency)
- Score/value for a word = TF * IDF

In [13]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are great pets",
    "the the the the the the the the"
]

In [14]:
[d.split() for d in corpus]

[['the', 'cat', 'sat', 'on', 'the', 'mat'],
 ['the', 'dog', 'sat', 'on', 'the', 'log'],
 ['cats', 'and', 'dogs', 'are', 'great', 'pets'],
 ['the', 'the', 'the', 'the', 'the', 'the', 'the', 'the']]

In [15]:
import math

In [16]:
N = len(corpus)
print(N)

docs = [d.split() for d in corpus]

def term_freq(doc, term):
    return doc.count(term)

def doc_freq(term):
    return sum(1 for d in docs if term in d)

def idf(term):
    df = doc_freq(term)
    return math.log((1+N)/(1+df))+1

def tfidf(term, doc):
    tf = term_freq(doc, term)
    return tf * idf(term)

4


In [17]:
final_dict = {}
for i, document in enumerate(corpus):
    final_dict[i] = {}
    for word in document.split():
        word_tfidf = tfidf(word, document)
        final_dict[i][word] = word_tfidf
        
print(final_dict)

{0: {'the': 2.4462871026284194, 'cat': 1.916290731874155, 'sat': 1.5108256237659907, 'on': 1.5108256237659907, 'mat': 1.916290731874155}, 1: {'the': 2.4462871026284194, 'dog': 1.916290731874155, 'sat': 1.5108256237659907, 'on': 1.5108256237659907, 'log': 1.916290731874155}, 2: {'cats': 1.916290731874155, 'and': 1.916290731874155, 'dogs': 1.916290731874155, 'are': 1.916290731874155, 'great': 1.916290731874155, 'pets': 1.916290731874155}, 3: {'the': 9.785148410513678}}


{0: {'the': 2.4462871026284194, 'cat': 1.916290731874155, 'sat': 1.5108256237659907, 'on': 1.5108256237659907, 'mat': 1.916290731874155}, 

1: {'the': 2.4462871026284194, 'dog': 1.916290731874155, 'sat': 1.5108256237659907, 'on': 1.5108256237659907, 'log': 1.916290731874155}, 

2: {'cats': 1.916290731874155, 'and': 1.916290731874155, 'dogs': 1.916290731874155, 'are': 1.916290731874155, 'great': 1.916290731874155, 'pets': 1.916290731874155}, 

3: {'the': 9.785148410513678}}


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

vocab = tfidf_vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vocab).round(3)
tfidf_df

,and,are,cat,cats,dog,dogs,great,log,mat,on,pets,sat,the
0,0.000,0.000,0.453,0.000,0.000,0.000,0.000,0.000,0.453,0.357,0.000,0.357,0.578
1,0.000,0.000,0.000,0.000,0.453,0.000,0.000,0.453,0.000,0.357,0.000,0.357,0.578
2,0.408,0.408,0.000,0.408,0.000,0.408,0.408,0.000,0.000,0.000,0.408,0.000,0.000
3,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000
